In [1]:
# %%

import warnings
import sys
import csv
import os
import pandas as pd

sys.path.append('/home/gli/gli-data-science/akhiyar/lib/')
if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore" # Also affect subprocesses


import os
import ds_db
import helper_db
from helper_db import read_bq
from helper import transform_to_rupiah, rupiah_format

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.express as px

pd.options.display.float_format = '{:,.3f}'.format
import plotly.graph_objects as go
import string
from nltk.corpus import stopwords
from wordcloud import WordCloud,STOPWORDS
import re

from nlp_id.tokenizer import Tokenizer 
tokenizer = Tokenizer()

from nlp_id.lemmatizer import Lemmatizer 
lemmatizer = Lemmatizer() 

### get list of stopword
from nlp_id.stopword import StopWord 
stopword = StopWord() 
li_stopword = stopword.get_stopword() 

sw_eng = stopwords.words("indonesian") + list(string.punctuation) + li_stopword


#from mlxtend.frequent_patterns import apriori, association_rules#
#import rarfile

In [2]:
# %%

df = pd.read_excel('/home/gli/gli-data-science/Frans/model/dataset1.xlsx', sheet_name='OKT-NOV')
df

,TANGGAL,BULAN,KODE_TOKO,JUMLAH_STRUK_SHIFT_PAGI,PERCENT_ONTIME_SHIFT_PAGI,AVG_SLA_SHIFT_PAGI,QTY_PER_STRUK_SHIFT_PAGI,SALES_SHIFT_PAGI,JUMLAH_STRUK_SHIFT_SIANG,PERCENT_ONTIME_SHIFT_SIANG,AVG_SLA_SHIFT_SIANG,QTY_PER_STRUK_SHIFT_SIANG,SALES_SHIFT_SIANG,KURIR_PAGI,KURIR_SIANG
0,1,10,1M6U,142,0.204,124.204,7.563,"15,144,114.560",148,0.365,79.306,6.612,"13,628,765.740",3,6
1,1,10,1M6V,166,0.163,138.476,10.988,"21,941,389.670",160,0.287,107.974,9.338,"16,598,755.790",3,4
2,1,10,JD75,187,0.289,90.984,8.027,"23,167,740.650",132,0.462,65.287,7.070,"12,373,117.150",5,4
3,1,10,JD76,92,1.000,29.174,10.707,"13,081,894.560",68,0.971,31.448,11.567,"8,977,878.370",4,4
4,1,10,KF18,143,0.650,53.154,11.245,"16,707,311.540",161,0.354,75.117,7.182,"11,379,151.400",2,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
807,31,10,TI12,143,0.119,195.476,8.601,"15,365,619.890",132,0.114,183.078,8.884,"12,882,834.210",5,6
808,31,10,TI17,233,0.206,144.871,8.940,"21,441,707.310",179,0.341,94.667,7.276,"13,455,720.620",6,5
809,31,10,UD92,159,0.151,139.767,8.793,"16,802,278.110",125,0.184,111.154,9.094,"11,546,145.030",4,4
810,31,10,UE21,107,1.000,26.907,8.906,"9,476,654.900",177,0.994,22.722,6.210,"12,102,683.840",5,4


In [3]:
# %%

df[df['KODE_TOKO'] == '1M6U'].sort_values(by=['BULAN', 'TANGGAL']).head(30)

,TANGGAL,BULAN,KODE_TOKO,JUMLAH_STRUK_SHIFT_PAGI,PERCENT_ONTIME_SHIFT_PAGI,AVG_SLA_SHIFT_PAGI,QTY_PER_STRUK_SHIFT_PAGI,SALES_SHIFT_PAGI,JUMLAH_STRUK_SHIFT_SIANG,PERCENT_ONTIME_SHIFT_SIANG,AVG_SLA_SHIFT_SIANG,QTY_PER_STRUK_SHIFT_SIANG,SALES_SHIFT_SIANG,KURIR_PAGI,KURIR_SIANG
0,1,10,1M6U,142,0.204,124.204,7.563,"15,144,114.560",148,0.365,79.306,6.612,"13,628,765.740",3,6
28,2,10,1M6U,125,0.336,96.800,8.712,"14,063,093.760",49,0.490,64.130,7.413,"4,564,065.120",5,5
56,3,10,1M6U,146,0.363,87.952,7.267,"14,825,449.600",114,0.465,76.690,5.965,"10,339,877.350",4,3
84,4,10,1M6U,147,0.360,89.156,9.007,"20,113,450.420",92,0.467,71.656,6.056,"7,530,965.810",3,5
112,5,10,1M6U,150,0.253,118.487,8.240,"14,363,115.510",118,0.381,83.518,7.746,"11,384,073.020",5,4
140,6,10,1M6U,143,0.161,141.133,10.413,"17,246,404.210",125,0.320,94.615,8.246,"11,208,222.440",4,3
168,7,10,1M6U,114,0.333,102.263,8.553,"14,017,701.050",116,0.578,56.167,5.702,"10,778,114.470",2,6
196,8,10,1M6U,112,0.482,70.000,6.295,"10,882,341.450",103,1.000,36.932,5.990,"9,681,593.740",4,5
224,9,10,1M6U,116,0.362,77.612,6.078,"10,794,298.370",127,0.409,69.724,7.236,"12,883,703.780",4,4
252,10,10,1M6U,136,0.338,100.765,5.831,"11,063,636.160",119,0.471,70.076,5.398,"10,240,687.430",3,4


In [4]:
# %%

# Ambil nilai unik di kolom 'Category' dan urutkan
unique_categories = df['KODE_TOKO'].unique()  # Ambil nilai unik
sorted_categories = sorted(unique_categories)  # Urutkan nilai unik

# Buat mapping berdasarkan urutan yang sudah di-sort
category_mapping = {category: index + 1 for index, category in enumerate(sorted_categories)}
print(category_mapping)
df['KODE_TOKO'] = df['KODE_TOKO'].map(category_mapping)
df

{'1M6U': 1, '1M6V': 2, 'JD75': 3, 'JD76': 4, 'KF18': 5, 'KF65': 6, 'PA39': 7, 'TH06': 8, 'TH72': 9, 'TI12': 10, 'TI17': 11, 'UD92': 12, 'UE21': 13, 'W785': 14}


,TANGGAL,BULAN,KODE_TOKO,JUMLAH_STRUK_SHIFT_PAGI,PERCENT_ONTIME_SHIFT_PAGI,AVG_SLA_SHIFT_PAGI,QTY_PER_STRUK_SHIFT_PAGI,SALES_SHIFT_PAGI,JUMLAH_STRUK_SHIFT_SIANG,PERCENT_ONTIME_SHIFT_SIANG,AVG_SLA_SHIFT_SIANG,QTY_PER_STRUK_SHIFT_SIANG,SALES_SHIFT_SIANG,KURIR_PAGI,KURIR_SIANG
0,1,10,1,142,0.204,124.204,7.563,"15,144,114.560",148,0.365,79.306,6.612,"13,628,765.740",3,6
1,1,10,2,166,0.163,138.476,10.988,"21,941,389.670",160,0.287,107.974,9.338,"16,598,755.790",3,4
2,1,10,3,187,0.289,90.984,8.027,"23,167,740.650",132,0.462,65.287,7.070,"12,373,117.150",5,4
3,1,10,4,92,1.000,29.174,10.707,"13,081,894.560",68,0.971,31.448,11.567,"8,977,878.370",4,4
4,1,10,5,143,0.650,53.154,11.245,"16,707,311.540",161,0.354,75.117,7.182,"11,379,151.400",2,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
807,31,10,10,143,0.119,195.476,8.601,"15,365,619.890",132,0.114,183.078,8.884,"12,882,834.210",5,6
808,31,10,11,233,0.206,144.871,8.940,"21,441,707.310",179,0.341,94.667,7.276,"13,455,720.620",6,5
809,31,10,12,159,0.151,139.767,8.793,"16,802,278.110",125,0.184,111.154,9.094,"11,546,145.030",4,4
810,31,10,13,107,1.000,26.907,8.906,"9,476,654.900",177,0.994,22.722,6.210,"12,102,683.840",5,4


In [5]:
# %%

df['jumlah_kurir'] = df['KURIR_PAGI'] + df['KURIR_SIANG']
df = df[[
    'TANGGAL', 'BULAN', 'KODE_TOKO', 'JUMLAH_STRUK_SHIFT_PAGI',
       'PERCENT_ONTIME_SHIFT_PAGI', 'AVG_SLA_SHIFT_PAGI',
       'QTY_PER_STRUK_SHIFT_PAGI', 'SALES_SHIFT_PAGI',
       'KURIR_PAGI','jumlah_kurir'
]]

In [6]:
# %%

X = df.drop(columns='KURIR_PAGI')
y = df[['KURIR_PAGI']]
y

,KURIR_PAGI
0,3
1,3
2,5
3,4
4,2
...,...
807,5
808,6
809,4
810,5


In [7]:
# %%

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
len(X_train), len(X_test)

(544, 268)

In [8]:
# %%

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

reg = XGBRegressor(random_state=42).fit(X_train, y_train)
reg.score(X, y)

0.8802918923636501

In [9]:
# %%

X_train.sample()

,TANGGAL,BULAN,KODE_TOKO,JUMLAH_STRUK_SHIFT_PAGI,PERCENT_ONTIME_SHIFT_PAGI,AVG_SLA_SHIFT_PAGI,QTY_PER_STRUK_SHIFT_PAGI,SALES_SHIFT_PAGI,jumlah_kurir
349,13,10,14,152,0.158,142.776,7.053,"11,761,226.550",6


In [10]:
# %%

def usulan_kurir(kode_toko_input, tanggal, bulan, ontime=1, avg_sla=60):
    kode_toko = category_mapping[kode_toko_input]

    import math
    data_pred = pd.DataFrame({
        'TANGGAL':[tanggal],
        'BULAN':[bulan],
        'KODE_TOKO':[kode_toko],
        'JUMLAH_STRUK_SHIFT_PAGI':[math.ceil(df[df['KODE_TOKO']== kode_toko].sort_values(by=['BULAN', 'TANGGAL'], ascending=[False, False]).head(10)['JUMLAH_STRUK_SHIFT_PAGI'].mean())],
        'PERCENT_ONTIME_SHIFT_PAGI':[ontime],
        'AVG_SLA_SHIFT_PAGI': [avg_sla],
        'QTY_PER_STRUK_SHIFT_PAGI':[math.ceil(df[df['KODE_TOKO']== kode_toko].sort_values(by=['BULAN', 'TANGGAL'], ascending=[False, False]).head(10)['QTY_PER_STRUK_SHIFT_PAGI'].mean())],
        'SALES_SHIFT_PAGI': [math.ceil(df[df['KODE_TOKO']== kode_toko].sort_values(by=['BULAN', 'TANGGAL'], ascending=[False, False]).head(10)['SALES_SHIFT_PAGI'].mean())],
        'jumlah_kurir':[math.ceil(df[df['KODE_TOKO']==kode_toko].sort_values(by=['BULAN', 'TANGGAL'], ascending=[False, False]).head(10)['jumlah_kurir'].mean())]

    })

    usulan_kurir_pagi = math.ceil(reg.predict(data_pred))
    usulan_kurir_siang = math.ceil(df[df['KODE_TOKO']==kode_toko].sort_values(by=['BULAN', 'TANGGAL'], ascending=[False, False]).head(10)['jumlah_kurir'].mean()) - usulan_kurir_pagi
    
    print('Usulan Kurir Pagi:', usulan_kurir_pagi)
    print('Usulan Kurir Siang:', usulan_kurir_siang)

    return usulan_kurir_pagi, usulan_kurir_siang

In [13]:
# %%

toko_input = 'TI12'
# usulan_kurir(kode_toko_input='1M6U', tanggal=1, bulan=12)
pagi, siang = usulan_kurir(kode_toko_input=toko_input, tanggal=9, bulan=12)

Usulan Kurir Pagi: 5
Usulan Kurir Siang: 4


In [15]:
# %%
# Proffing
test = df[df['KODE_TOKO']==category_mapping[toko_input]].sort_values(by=['BULAN', 'TANGGAL'], ascending=[False, False]).head(30)
print('rata2 ontime rate: ',test[test['jumlah_kurir'] >= pagi + siang]['PERCENT_ONTIME_SHIFT_PAGI'].mean(),
      '\nrata2 sales: ', test[test['jumlah_kurir'] >= pagi + siang]['SALES_SHIFT_PAGI'].mean(),
      '\nrata2 sla: ', test[test['jumlah_kurir'] >= pagi + siang]['AVG_SLA_SHIFT_PAGI'].mean(),
      )
test[test['jumlah_kurir'] >= pagi + siang]

rata2 ontime rate:  0.5822499999999999 
rata2 sales:  13912675.599444441 
rata2 sla:  72.65295769888888


,TANGGAL,BULAN,KODE_TOKO,JUMLAH_STRUK_SHIFT_PAGI,PERCENT_ONTIME_SHIFT_PAGI,AVG_SLA_SHIFT_PAGI,QTY_PER_STRUK_SHIFT_PAGI,SALES_SHIFT_PAGI,KURIR_PAGI,jumlah_kurir
765,28,11,10,179,0.592,58.391,7.911,"18,608,777.390",5,10
723,26,11,10,195,0.580,57.503,9.082,"21,410,482.920",5,9
695,25,11,10,169,0.680,49.893,8.319,"17,271,140.740",4,9
667,24,11,10,98,1.000,20.592,6.939,"8,527,596.490",6,9
639,23,11,10,130,0.885,32.477,6.531,"9,690,037.760",5,9
583,21,11,10,143,0.587,69.336,6.958,"12,097,565.990",5,10
499,18,11,10,145,0.517,63.828,8.083,"13,669,326.230",4,9
443,16,11,10,163,0.466,85.883,8.454,"16,345,337.030",6,10
415,15,11,10,130,0.777,45.246,7.831,"12,219,166.770",4,10
387,14,11,10,120,0.767,43.692,7.508,"10,234,154.250",4,10
